> 在线讲义：[第 17 章 · 训练与强化学习的计算和状态：为什么训练要 6ND、要存 18 字节](https://wikieden.github.io/ai-infra-from-constraints/lessons/17-training-rl.html) · 本书《从约束推导 AI Infra》

# 第 17 章 · 训练与强化学习的计算和状态：为什么训练要 6ND、要存 18 字节

配套讲义：仓库里 `ai-infra/notes/lessons/17-training-rl.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 三次矩阵乘法 | $2mkn$ × 3 | `linear_train_flops(2, 3, 4)` | 48、48、48 |
| 第二步 18 字节 | 逐项相加 | `train_state_gb(8_190_735_360)` | 147.433 GB = 137.308 GiB |
| 第三步 长度与配对 | 三角形数 | `doc_pairs([4096, 4096])`、`doc_pairs([7168, 1024])` | 16,781,312 和 26,218,496 |
| 第三步 6ND 修正 | 比值 | `431.368 / 402.591` | 高约 7.15% |
| 第四步 RL 周期 | $n(f_g + f_r) + F_u$ | `rl_cycle(n)` | 32 条 2,181.559，64 条 3,410.701 |
| 第四步 同步 | $rM_W$ | `sync_gb(4)` | 65.526 GB |

例题 3-3（Qwen3-8B 8,190,735,360 个参数）、长度分组、8192 token 训练矩阵运算、例题 3-4 的各阶段运算量取自 AII 第 3 章 §3.4–§3.5；
m = 2、k = 3、n = 4 是编的。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
N_QWEN = 8_190_735_360
F_PER_RESP = (465.143 + 129.056 + 634.944) / 32  # 一条回答的 prefill + decode + 参考模型前向
F_UPDATE = 952.416  # 16 条保留样本的一次策略更新

**函数 `linear_train_flops`**：(前向, 输入梯度, 权重梯度)，各 2mkn。

In [ ]:
def linear_train_flops(m: int, k: int, n: int) -> tuple[int, int, int]:
    """(前向, 输入梯度, 权重梯度)，各 2mkn。"""
    f = 2 * m * k * n
    return f, f, f

**函数 `train_state_gb`**：参数相关的长期训练状态：(GB, GiB)。默认 BF16 权重 2 + FP32 梯度 / 主权重 / Adam 两矩各 4。

In [ ]:
def train_state_gb(n_params: int, bytes_per: int = 18) -> tuple[float, float]:
    """参数相关的长期训练状态：(GB, GiB)。默认 BF16 权重 2 + FP32 梯度 / 主权重 / Adam 两矩各 4。"""
    b = n_params * bytes_per
    return b / 1e9, b / 2**30

**函数 `doc_pairs`**：若干篇独立文档的有效因果配对数：每篇一个三角形 P(P+1)/2。

In [ ]:
def doc_pairs(lengths: list[int]) -> int:
    """若干篇独立文档的有效因果配对数：每篇一个三角形 P(P+1)/2。"""
    return sum(p * (p + 1) // 2 for p in lengths)

**函数 `rl_cycle`**：RL 一轮的矩阵运算（TFLOPs）= 生成数 × (生成 + 参考模型) + 固定更新。

In [ ]:
def rl_cycle(n_generate: int) -> float:
    """RL 一轮的矩阵运算（TFLOPs）= 生成数 × (生成 + 参考模型) + 固定更新。"""
    return n_generate * F_PER_RESP + F_UPDATE

**函数 `sync_gb`**：每个生成副本独立收一份完整权重的总传输量。

In [ ]:
def sync_gb(replicas: int, weight_gb: float = 16.38147072) -> float:
    """每个生成副本独立收一份完整权重的总传输量。"""
    return replicas * weight_gb

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：线性层训练（编的 m=2、k=3、n=4）")
print(f"  前向 / 输入梯度 / 权重梯度 = {linear_train_flops(2, 3, 4)}，合计 {sum(linear_train_flops(2, 3, 4))}")

In [ ]:
print("\n第二步：Qwen3-8B 全参数训练的参数相关状态")
gb, gib = train_state_gb(N_QWEN)
print(f"  18 B × 参数 = {gb:.3f} GB = {gib:.3f} GiB（推理 BF16 权重的 9 倍）")

In [ ]:
print("\n第三步：同样 8192 个 token，分组不同")
for g in ([4096, 4096], [7168, 1024]):
    pairs = doc_pairs(g)
    print(f"  {g}：配对 {pairs:,}，一层注意力前向 {16384 * pairs / 1e12:.3f} TFLOPs")
print(f"  逐项 431.368 vs 6ND 402.591：高 {431.368 / 402.591 - 1:.2%}")

In [ ]:
print("\n第四步：RL 一轮（保留 16 条）")
for n in (32, 64):
    c = rl_cycle(n)
    print(f"  生成 {n} 条：{c:,.3f} TFLOPs，每个保留样本 {c / 16:.3f}")
print(f"  4 个副本同步权重：{sync_gb(4):.3f} GB")

## 3. 代码里才看得到的

1. `rl_cycle(64)` 算出 3,410.702，和原文 3,410.701 只差 0.001（各阶段只给到三位小数）：说明例题里各阶段确实严格按回答数线性增长（等长样本、不提前停止、不共享前缀）。
2. `doc_pairs` 对每篇文档单独算三角形：拼成一条 8192 的长序列又不加文档边界掩码的话，配对会变成 33,558,528，比两篇独立文档多一倍。
3. 18 字节里没有激活；激活随序列长度和层数变，原书第 10 章另算。

## 4. 自己改一改

- 用 `doc_pairs` 比较 8 × 1024 和 1 × 8192 两种切法的注意力配对。
- 把 `train_state_gb` 的每参数字节改成 16（梯度 BF16），看能省多少。
- 画出 `rl_cycle(n) / 16` 随 $n$ 的变化：每个保留样本的成本怎么随保留比例下降而上升。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 复盘要草稿，倒推用方法：前向一次，反向两次；
> 账本 2 字节，草稿总账两本笔记各 4 字节；
> 人头估餐费，漏了酒水多算了旁观；
> 题筛得越严，做得越多，总结不变。